# 04 v2. 라벨 다시 정리 → 문장 → 분할 (A, B)

`04_split_local.ipynb`를 고친 버전. 기존 결과(`data/splits/`)는 그대로 두고 새 결과는 `data/v2/`에 저장한다.

**04_split_local과 달라진 라벨 규칙**

| | 04_split_local | v2 |
|---|---|---|
| 제외 라벨 (Exfiltration, Execution, Lateral Movement) | 붙은 윈도우를 삭제 | **살림**. 데이터가 적어 학습은 못 하므로 숨긴 라벨처럼 시험에서 제로샷으로 평가 (`role = held_out`) |
| 같은 내용(signature)인데 라벨이 다름 | 팀 라벨이 다르면 전부 삭제, 기법만 다르면 첫 윈도우의 기법만 남김 | **합집합으로 합침** (전술·기법 모두). 같은 패턴이 여러 라벨일 수 있다는 걸 그대로 학습 |
| 같은 내용인데 정상과 공격 | 삭제 | **`{BENIGN, 공격}`으로 합침**. "정상일 수도, 공격일 수도 있다"를 학습 (GLiClass는 라벨마다 따로 sigmoid) |
| 전술 라벨 | 팀이 묶은 라벨 (예: `TA0001+TA0005`) | **MITRE 전술 각각** (TA0001, TA0003, TA0004, TA0005 ...). 해당하는 전술을 전부 붙이는 멀티라벨 |
| 윈도우 라벨 | 전술만 | **전술과 기법을 둘 다** 기록 (`labels`, `techniques`) |

| 기법 ID | 상위 기법(T1078 등)과 사용자 정의 ID(T1552.GEN)가 섞여 있음 | 확실한 것은 하위 기법으로 고치고, MITRE에 없는 ID는 상위 MITRE 기법으로 바꿈 (1-2번) |
| 기법 라벨 | 이벤트의 기법 하나만 | **하위 기법이면 상위 기법도 함께** 멀티라벨 (예: T1078.004 → `T1078`, `T1078.004`) |

**진행 순서**: 1. 전술 라벨 표 → 1-2. 기법 고치기 → 2. 윈도우 라벨 → 3. 같은 내용 합치기 (이 노트북은 여기까지 먼저 확인)

## 0. 경로

In [1]:
import os, json, shutil
from pathlib import Path
import pandas as pd, numpy as np
from collections import Counter
pd.set_option('display.width', 220); pd.set_option('display.max_colwidth', 120)

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
INTERIM = REPO / 'data' / 'interim'
OUT = REPO / 'data' / 'v2'
OUT.mkdir(parents=True, exist_ok=True)
print('OUT =', OUT)

OUT = /home/openclaw/work2/ctgli/data/v2


## 1. 전술 라벨 표 (MITRE 전술 각각)

[하는 일] MITRE 전술 12개 + Benign을 라벨 하나씩으로 둔다. 라벨 문장은 v19 표의 `이름: 설명` (Benign은 팀 문장).
역할은 팀 `labels/labels.csv`를 따른다.
- 팀의 학습 라벨에 속한 전술 → `train` (묶여 있던 TA0001+TA0005, TA0003+TA0004는 각각 따로)
- 팀의 숨김(`held_out`) + 제외(`excluded`) 라벨 → `held_out` (시험 전용, 제로샷). 제외 라벨을 살림

결과는 `data/v2/labels.csv`. 팀 원본 파일은 건드리지 않는다.

[볼 것] 학습 7개 (전술 6 + Benign), 시험 전용 6개.

In [2]:
import io, subprocess
team = pd.read_csv(REPO / 'labels' / 'labels.csv')
team_role = {t: r.role for r in team.itertuples() for t in str(r.source_tactics).split('|') if t and t != 'nan'}
# v19 라벨 표 (팀이 저장소에서 지운 파일이라 git 기록에서 읽음)
v19 = pd.read_csv(io.StringIO(subprocess.run(['git', '-C', str(REPO), 'show', '74782a0:labels/attack_labels_v19.csv'],
                                              capture_output=True, text=True, check=True).stdout)).set_index('node_id')

tactics = [t for t in v19[v19.level == 'top'].index if t != 'BENIGN']
rows = [dict(label_id=t, name=v19.at[t, 'name'], role_team=team_role[t],
             role='train' if team_role[t] == 'train' else 'held_out',
             label_text=f"{v19.at[t, 'name']}: {v19.at[t, 'desc']}") for t in tactics]
rows.append(dict(label_id='BENIGN', name='Benign', role_team='train', role='train',
                 label_text=team.set_index('label_id').at['BENIGN', 'label_text']))
labels = pd.DataFrame(rows)
labels.to_csv(OUT / 'labels.csv', index=False)

TRAIN = labels[labels.role == 'train'].label_id.tolist()
HELD = labels[labels.role == 'held_out'].label_id.tolist()
name = dict(zip(labels.label_id, labels.name))
print(labels[['label_id', 'name', 'role_team', 'role', 'label_text']].to_string(index=False))

label_id                 name role_team     role                                                                                                                                                                                                                                                                                                                                                                                                                  label_text
  TA0001       Initial Access     train    train                                                                                                                                                                                                                                                                                                                                 Initial Access: getting first access to the cloud account with valid or trusted credentials
  TA0002            Execution  excluded held_out                              

## 1-2. 기법 고치기 (이벤트 단위)

[하는 일] 이벤트의 기법을 고친 뒤, 전술은 고친 기법에서 다시 구한다 (v19 표의 상위 관계).

1. **상위 기법 → 하위 기법**: Splunk 시나리오가 상위 기법(T1078)으로만 라벨을 줬지만, 이벤트를 보면 하위 기법이 분명한 것
2. **MITRE에 없는 ID → 상위 MITRE 기법**: `T1552.GEN`(v19 표에서 만든 사용자 정의 ID) → `T1552`

맞는 하위 기법이 없는 상위 기법(T1098 삭제, T1110, T1552, T1485)은 그대로 둔다. 상위 기법도 MITRE ID다.

[볼 것] 고친 이벤트 수, 고친 뒤 상위 기법이 남은 곳.

In [3]:
# (세션, API) → 새 기법. 이벤트를 보고 확실한 것만
FIX = {
    ('splunk:assume_role_with_saml', 'AssumeRoleWithSAML'): 'T1078.004',        # SAML로 클라우드 계정 로그인
    ('splunk:aws_createloginprofile', 'ConsoleLogin'): 'T1078.004',             # 콘솔 로그인
    ('splunk:aws_createaccesskey', 'CreateAccessKey'): 'T1098.001',             # 다른 사용자에게 액세스 키 발급
    ('splunk:aws_createloginprofile', 'CreateLoginProfile'): 'T1098.001',       # 로그인 비밀번호 발급
    ('splunk:aws_updateloginprofile', 'UpdateLoginProfile'): 'T1098.001',       # 로그인 비밀번호 변경
    ('splunk:aws_create_policy_version', 'CreatePolicyVersion'): 'T1098.003',   # 정책 버전을 바꿔 권한 확대
    ('splunk:aws_setdefaultpolicyversion', 'SetDefaultPolicyVersion'): 'T1098.003',
}
NOT_MITRE = {'T1552.GEN': 'T1552'}                                              # v19 표의 사용자 정의 ID

def with_parents(t):
    """기법과 그 상위 기법들 (전술은 제외). 예: T1098.001 → {T1098, T1098.001}"""
    if v19.at[t, 'level'] == 'top':
        return set()
    return {t}.union(*[with_parents(p) for p in str(v19.at[t, 'parent_ids']).split('|')])

def tactics_of(t):
    if v19.at[t, 'level'] == 'top':
        return {t}
    return set().union(*[tactics_of(p) for p in str(v19.at[t, 'parent_ids']).split('|')])

events = {}
for k in 'AB':
    e = pd.read_csv(INTERIM / f'events_{k}.csv', low_memory=False)
    e['is_attack'] = e.is_attack.astype(str) == 'True'
    old = e.technique.copy()
    fixed = [FIX.get((s, n), t) for s, n, t in zip(e.session, e.eventName, e.technique)]
    e['technique'] = [NOT_MITRE.get(t, t) if isinstance(t, str) else t for t in fixed]
    e['tactics'] = [('|'.join(sorted(tactics_of(t))) if isinstance(t, str) else '') for t in e.technique]
    ch = e[e.is_attack & (old != e.technique)]
    print(f'[{k}] 기법을 고친 이벤트 {len(ch)}:', ch.groupby([old[ch.index], ch.technique]).size().to_dict())
    events[k] = e.set_index('event_uid')

left = events['A'][events['A'].is_attack].technique
parent = sorted(t for t in left.unique() if v19.at[t, 'level'] == 'mid' and any(str(v19.at[c, 'parent_ids']) == t for c in v19.index))
print('하위 기법이 있는데 상위로 남은 기법:', {t: int((left == t).sum()) for t in parent})
print('MITRE에 없는 ID 남음:', sorted(set(left) & set(v19[v19.is_custom == 'yes'].index)))

[A] 기법을 고친 이벤트 179: {('T1078', 'T1078.004'): 62, ('T1078', 'T1098.001'): 58, ('T1078', 'T1098.003'): 6, ('T1552.GEN', 'T1552'): 53}
[B] 기법을 고친 이벤트 105: {('T1078', 'T1078.004'): 47, ('T1078', 'T1098.001'): 46, ('T1078', 'T1098.003'): 6, ('T1552.GEN', 'T1552'): 6}
하위 기법이 있는데 상위로 남은 기법: {'T1098': 245, 'T1110': 9, 'T1485': 1, 'T1552': 113}
MITRE에 없는 ID 남음: []


## 2. 윈도우 라벨 (전술 + 기법)

[하는 일] 윈도우의 이벤트들(1-2번에서 고친 라벨)로 방식 1 라벨을 다시 계산한다. 공격 이벤트가 하나라도 있으면 그 이벤트들의
- `labels`: MITRE 전술 각각 (예: T1078이면 `TA0001`, `TA0003`, `TA0004`, `TA0005` 네 개). 공격 이벤트가 없으면 `BENIGN`
- `techniques`: 기법 + 그 상위 기법 (예: `T1098.001`이면 `T1098`, `T1098.001` 두 개). 공격 이벤트가 없으면 `BENIGN`

[볼 것] 지우는 윈도우가 없으므로 개수가 `windows_*.csv`와 같아야 함.

In [4]:
def window_labels(k):
    w = pd.read_csv(INTERIM / f'windows_{k}_size5_stride1.csv', low_memory=False, keep_default_na=False)
    e = events[k]
    ev = w[['event_uids']].assign(uid=w.event_uids.str.split('|')).explode('uid')
    ev = ev.join(e[['is_attack', 'technique', 'tactics']], on='uid')
    ev = ev[ev.is_attack]
    tech = ev.groupby(level=0).technique.agg(lambda s: sorted(set().union(*[with_parents(t) for t in s])))
    tac = ev.groupby(level=0).tactics.agg(lambda s: sorted({t for x in s for t in x.split('|') if t}))
    w['techniques'] = tech.reindex(w.index).apply(lambda x: x if isinstance(x, list) else ['BENIGN'])
    w['labels'] = tac.reindex(w.index).apply(lambda x: x if isinstance(x, list) else ['BENIGN'])
    assert (w['labels'].apply(lambda l: l != ['BENIGN']) == (w.m1_is_attack.astype(str) == 'True')).all()   # 공격/정상 판정은 그대로
    return w

wins = {k: window_labels(k) for k in 'AB'}
for k, w in wins.items():
    revived = w['labels'].apply(lambda ls: any(team_role.get(l) == 'excluded' for l in ls))
    n_tac = w[w['labels'].apply(lambda ls: ls != ['BENIGN'])]['labels'].map(len).value_counts().sort_index().to_dict()
    print(f'[{k}] 윈도우 {len(w)} | 살린 제외 전술 윈도우 {int(revived.sum())} | 공격 윈도우의 전술 개수 분포 {n_tac}')

[A] 윈도우 28387 | 살린 제외 전술 윈도우 99 | 공격 윈도우의 전술 개수 분포 {1: 8149, 2: 386, 4: 64, 5: 26, 6: 32}
[B] 윈도우 8183 | 살린 제외 전술 윈도우 14 | 공격 윈도우의 전술 개수 분포 {1: 3100, 2: 298, 4: 47, 5: 21, 6: 36}


## 3. 같은 내용(signature) 합치기

[하는 일] signature가 같은 윈도우를 하나로 합치고 라벨은 합집합으로 붙인다.
- 전술(`labels`)과 기법(`techniques`) 각각 합집합
- 대표 윈도우는 처음 나온 것 (세션, 윈도우 번호 순). 분할 때 이 윈도우의 세션·위치를 쓴다
- `n_merged`: 합친 윈도우 수, `n_sessions`: 나온 세션 수, `ambiguous`: 정상과 공격이 함께 붙은 경우

[볼 것] 라벨이 늘어난 윈도우 수, `{BENIGN, 공격}` 윈도우 수.

[확인 질문] 합집합 라벨은 "둘 다 맞다"가 아니라 "둘 다 가능하다"는 뜻이다. 평가할 때 이걸 어떻게 해석해야 할까?

In [5]:
def union(series):
    s = sorted(set(l for ls in series for l in ls))
    return s

def merge_same(w):
    w = w.sort_values(['session', 'window'], kind='stable')
    g = w.groupby('signature', sort=False)
    first = g.head(1).set_index('signature')
    first['labels'] = g['labels'].apply(union)
    first['techniques'] = g['techniques'].apply(union)
    first['n_merged'] = g.size()
    first['n_sessions'] = g.session.nunique()
    first['label_changed'] = g['labels'].apply(lambda s: len(set(map(tuple, s))) > 1)
    first['tech_changed'] = g['techniques'].apply(lambda s: len(set(map(tuple, s))) > 1)
    first['ambiguous'] = first['labels'].apply(lambda ls: 'BENIGN' in ls and len(ls) > 1)
    return first.reset_index()

merged = {k: merge_same(w) for k, w in wins.items()}
for k, m in merged.items():
    print(f'[{k}] {len(wins[k])} → {len(m)} 윈도우 | 전술 라벨이 합쳐진 {int(m.label_changed.sum())} | 기법이 합쳐진 {int(m.tech_changed.sum())} | '
          f'{{BENIGN, 공격}} {int(m.ambiguous.sum())}')

[A] 28387 → 9293 윈도우 | 전술 라벨이 합쳐진 9 | 기법이 합쳐진 83 | {BENIGN, 공격} 1
[B] 8183 → 7737 윈도우 | 전술 라벨이 합쳐진 1 | 기법이 합쳐진 46 | {BENIGN, 공격} 0


[볼 것] 합쳐져서 라벨이 여러 개가 된 예

In [6]:
for k, m in merged.items():
    print(f'===== {k}')
    print('전술 합집합 예:', m[m.label_changed]['labels'].map('|'.join).value_counts().head(6).to_dict())
    print('기법 합집합 예:', m[m.tech_changed]['techniques'].map('|'.join).value_counts().head(6).to_dict())
    print('정상+공격:', m[m.ambiguous][['labels', 'techniques', 'n_merged', 'n_sessions']].to_dict('records'))

===== A
전술 합집합 예: {'TA0002|TA0006': 7, 'BENIGN|TA0001|TA0003|TA0004|TA0005': 1, 'TA0001|TA0003|TA0004|TA0005|TA0006': 1}
기법 합집합 예: {'T1526|T1580': 74, 'T1552|T1552.005|T1651': 7, 'BENIGN|T1078|T1078.004': 1, 'T1078|T1078.004|T1110|T1110.003': 1}
정상+공격: [{'labels': ['BENIGN', 'TA0001', 'TA0003', 'TA0004', 'TA0005'], 'techniques': ['BENIGN', 'T1078', 'T1078.004'], 'n_merged': 4, 'n_sessions': 1}]
===== B
전술 합집합 예: {'TA0001|TA0003|TA0004|TA0005|TA0006': 1}
기법 합집합 예: {'T1526|T1580': 45, 'T1078|T1078.004|T1110|T1110.003': 1}
정상+공격: []


## 4. 라벨별 개수 (합친 뒤)

[볼 것] 데이터에 있는 전술과 기법이 전부 남아야 함 (지운 것 없음). Stealth(TA0005)는 v19 기준 기법이 T1078 계열뿐이라 Initial Access와 개수가 같음.

In [7]:
rows = []
for k, m in merged.items():
    c = m.explode('labels')['labels'].value_counts()
    for l in labels.label_id:
        rows.append({'data': k, 'label_id': l, 'name': name[l], 'role': labels.set_index('label_id').at[l, 'role'],
                     'windows': int(c.get(l, 0)), 'sessions': int(m[m['labels'].apply(lambda ls: l in ls)].session.nunique())})
t = pd.DataFrame(rows).pivot_table(index=['label_id', 'name', 'role'], columns='data', values=['windows', 'sessions']).astype(int)
print(t.sort_values(('windows', 'B'), ascending=False).to_string())

tech = pd.DataFrame({k: m.explode('techniques')['techniques'].value_counts() for k, m in merged.items()}).fillna(0).astype(int)
print('\n기법', len(tech), '종 (BENIGN 포함)'); print(tech.sort_values('B', ascending=False).to_string())

                                       sessions     windows      
data                                          A   B       A     B
label_id name                 role                               
BENIGN   Benign               train           3   3    5751  4495
TA0007   Discovery            train           7   7    2837  2520
TA0040   Impact               held_out        6   6     493   453
TA0003   Persistence          train          21  21     224   291
TA0004   Privilege Escalation train          20  20     223   290
TA0005   Stealth              train           5   5      77    90
TA0001   Initial Access       train           5   5      77    90
TA0112   Defense Impairment   held_out       17  17      73    70
TA0009   Collection           held_out        3   3      26    49
TA0006   Credential Access    train          12  12      39    13
TA0010   Exfiltration         held_out        6   6      10     7
TA0002   Execution            held_out        4   4      15     4
TA0008   L